In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%pip install duckdb

In [ ]:
import duckdb

print("DuckDB version:", duckdb.__version__)

con = duckdb.connect(":memory:")

result = con.execute("""
    SELECT
        'DuckDB is ready' AS status,
        1 + 1 AS test_result
""").fetchdf()

display(result)

In [ ]:

import duckdb

con = duckdb.connect(":memory:")

# Aktifkan dukungan akses file melalui HTTP.
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

source_url = (
    "https://huggingface.co/datasets/"
    "openfoodfacts/product-database/resolve/main/food.parquet"
)

# Periksa struktur kolom tanpa meminta seluruh isi tabel.
schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{source_url}')"
).fetchdf()

display(schema.head(20))

print("Jumlah kolom:", len(schema))

In [ ]:

source_url = (
    "https://huggingface.co/datasets/"
    "openfoodfacts/product-database/resolve/main/food.parquet"
)

sample = con.execute("""
    SELECT
        code,
        brands,
        brands_tags,
        categories,
        categories_tags,
        countries_tags,
        completeness
    FROM read_parquet(?)
    LIMIT 20
""", [source_url]).fetchdf()

display(sample)


In [ ]:

import duckdb

print("DuckDB version:", duckdb.__version__)

con = duckdb.connect(":memory:")

print("Koneksi DuckDB berhasil dibuat.")


In [ ]:

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

source_url = (
    "https://huggingface.co/datasets/"
    "openfoodfacts/product-database/resolve/main/food.parquet"
)

print("Akses Parquet siap diuji.")


In [ ]:

test = con.execute("""
    SELECT code, brands, countries_tags
    FROM read_parquet(?)
    LIMIT 5
""", [source_url]).fetchdf()

display(test)


In [ ]:

sample_profile = con.execute("""
    SELECT
        COUNT(*) AS sample_rows,
        COUNT(code) AS non_null_codes,
        COUNT(DISTINCT code) AS distinct_codes,
        COUNT(brands) AS non_null_brands,
        COUNT(categories_tags) AS non_null_category_tags,
        COUNT(countries_tags) AS non_null_country_tags,
        COUNT(completeness) AS non_null_completeness
    FROM (
        SELECT
            code,
            brands,
            categories_tags,
            countries_tags,
            completeness
        FROM read_parquet(?)
        LIMIT 1000
    ) AS sample
""", [source_url]).fetchdf()

display(sample_profile)


In [ ]:

country_sample = con.execute("""
    SELECT
        countries_tags,
        COUNT(*) AS product_records
    FROM (
        SELECT countries_tags
        FROM read_parquet(?)
        LIMIT 1000
    ) AS sample
    GROUP BY countries_tags
    ORDER BY product_records DESC
    LIMIT 15
""", [source_url]).fetchdf()

display(country_sample)


In [ ]:

country_quality = con.execute("""
    SELECT
        COUNT(*) AS sample_rows,

        SUM(
            CASE
                WHEN countries_tags = []
                THEN 1 ELSE 0
            END
        ) AS empty_country_arrays,

        SUM(
            CASE
                WHEN list_contains(countries_tags, 'en:null')
                THEN 1 ELSE 0
            END
        ) AS contains_null_tag,

        SUM(
            CASE
                WHEN list_contains(countries_tags, 'en:world')
                THEN 1 ELSE 0
            END
        ) AS contains_world_tag,

        SUM(
            CASE
                WHEN array_length(countries_tags) > 1
                THEN 1 ELSE 0
            END
        ) AS multiple_country_tags

    FROM (
        SELECT countries_tags
        FROM read_parquet(?)
        LIMIT 1000
    ) AS sample
""", [source_url]).fetchdf()

display(country_quality)


In [ ]:

indonesia_sample = con.execute("""
    SELECT
        COUNT(*) AS tagged_indonesia_records,
        COUNT(DISTINCT code) AS distinct_product_codes
    FROM (
        SELECT
            code,
            countries_tags
        FROM read_parquet(?)
        LIMIT 1000
    ) AS sample
    WHERE list_contains(countries_tags, 'en:indonesia')
""", [source_url]).fetchdf()

display(indonesia_sample)


In [ ]:

parquet_info = con.execute("""
    SELECT *
    FROM parquet_file_metadata(?)
""", [source_url]).fetchdf()

display(parquet_info)


In [ ]:

row_group_stats = con.execute("""
    SELECT *
    FROM parquet_metadata(?)
    WHERE path_in_schema = 'countries_tags'
    LIMIT 10
""", [source_url]).fetchdf()

display(row_group_stats)


In [ ]:

metadata_columns = con.execute("""
    SELECT DISTINCT path_in_schema
    FROM parquet_metadata(?)
    ORDER BY path_in_schema
    LIMIT 30
""", [source_url]).fetchdf()

display(metadata_columns)


In [ ]:

import requests

api_url = "https://world.openfoodfacts.org/api/v2/search"

params = {
    "countries_tags_en": "indonesia",
    "page_size": 1,
    "fields": "code,brands,categories_tags,countries_tags,completeness"
}

try:
    response = requests.get(
        api_url,
        params=params,
        headers={"User-Agent": "OFF-Brand-Landscape-Portfolio/1.0"},
        timeout=20
    )

    print("HTTP status:", response.status_code)
    response.raise_for_status()

    result = response.json()

    print("Total matching records reported by API:", result.get("count"))
    print("Records returned:", len(result.get("products", [])))

    if result.get("products"):
        display(result["products"])

except requests.exceptions.Timeout:
    print("Request timed out after 20 seconds.")

except requests.exceptions.RequestException as error:
    print("Request failed:", error)


In [ ]:

import shutil

disk = shutil.disk_usage("/kaggle/working")

print(f"Total disk: {disk.total / (1024**3):.2f} GB")
print(f"Used disk: {disk.used / (1024**3):.2f} GB")
print(f"Free disk: {disk.free / (1024**3):.2f} GB")


In [ ]:

import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")


In [ ]:

import time

start = time.time()

try:
    test = con.execute("""
        SELECT code, brands, countries_tags
        FROM read_parquet(?)
        LIMIT 1000
    """, [source_url]).fetchdf()

    print("Rows returned:", len(test))
    print(f"Elapsed seconds: {time.time() - start:.1f}")

except Exception as e:
    print(type(e).__name__, str(e))


In [ ]:

import duckdb
import time

con = duckdb.connect(":memory:")

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

source_url = (
    "https://huggingface.co/datasets/"
    "openfoodfacts/product-database/resolve/main/food.parquet"
)

print("DuckDB:", duckdb.__version__)
print("Connection ready:", con is not None)
print("Source URL ready:", source_url)


In [ ]:

start = time.time()

metadata = con.execute("""
    SELECT
        COUNT(*) AS metadata_entries,
        COUNT(DISTINCT row_group_id) AS row_groups
    FROM parquet_metadata(?)
""", [source_url]).fetchdf()

display(metadata)
print(f"Elapsed seconds: {time.time() - start:.1f}")


In [ ]:

import requests

response = requests.head(
    source_url,
    headers={"User-Agent": "OFF-Brand-Landscape-Portfolio/1.0"},
    allow_redirects=True,
    timeout=20
)

print("HTTP status:", response.status_code)

size = response.headers.get("Content-Length")
if size is not None:
    print(f"File size: {int(size) / (1024**3):.2f} GiB")
else:
    print("File size: not provided by server")

print("Accept-Ranges:", response.headers.get("Accept-Ranges"))


In [ ]:

import os
import requests
from IPython.display import display, clear_output

local_path = "/kaggle/working/food.parquet"
expected_size = int(response.headers["Content-Length"])
chunk_size = 8 * 1024 * 1024

downloaded = 0

with requests.get(
    source_url,
    headers={"User-Agent": "OFF-Brand-Landscape-Portfolio/1.0"},
    stream=True,
    timeout=(20, 120)
) as r:
    r.raise_for_status()

    with open(local_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

                if downloaded % (256 * 1024 * 1024) < chunk_size:
                    print(
                        f"Downloaded: {downloaded / (1024**3):.2f} / "
                        f"{expected_size / (1024**3):.2f} GiB"
                    )

actual_size = os.path.getsize(local_path)

print("\nDownload finished.")
print(f"Expected size: {expected_size:,} bytes")
print(f"Actual size:   {actual_size:,} bytes")
print(f"Size matches:  {actual_size == expected_size}")


In [ ]:

import os
import duckdb
import pandas as pd

local_path = "/kaggle/working/food.parquet"

assert os.path.exists(local_path), "File tidak ditemukan."
assert os.path.getsize(local_path) == 7_975_461_945, "Ukuran file tidak sesuai."

con = duckdb.connect(":memory:")

# 1. Periksa metadata file Parquet
file_meta = con.execute(
    "SELECT * FROM parquet_file_metadata(?)",
    [local_path]
).fetchdf()

# 2. Periksa struktur kolom
schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [local_path]
).fetchdf()

# 3. Hitung jumlah baris yang dapat dibaca
row_count = con.execute(
    "SELECT COUNT(*) AS total_rows FROM read_parquet(?)",
    [local_path]
).fetchone()[0]

print("File size:", f"{os.path.getsize(local_path):,}", "bytes")
print("Total rows:", f"{row_count:,}")
print("Total columns:", len(schema))
print("\nSchema preview:")
display(schema.head(20))


In [ ]:

import pandas as pd
from IPython.display import display

profile = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(code) AS non_null_codes,
        COUNT(DISTINCT code) AS distinct_non_null_codes,
        COUNT(*) FILTER (
            WHERE code IS NULL OR TRIM(code) = ''
        ) AS missing_or_empty_codes,

        COUNT(*) FILTER (
            WHERE brands IS NULL OR TRIM(brands) = ''
        ) AS missing_or_empty_brands,

        COUNT(*) FILTER (
            WHERE brands_tags IS NULL
        ) AS null_brand_tag_arrays,
        COUNT(*) FILTER (
            WHERE brands_tags IS NOT NULL
              AND len(brands_tags) = 0
        ) AS empty_brand_tag_arrays,

        COUNT(*) FILTER (
            WHERE categories_tags IS NULL
        ) AS null_category_tag_arrays,
        COUNT(*) FILTER (
            WHERE categories_tags IS NOT NULL
              AND len(categories_tags) = 0
        ) AS empty_category_tag_arrays,

        COUNT(*) FILTER (
            WHERE countries_tags IS NULL
        ) AS null_country_tag_arrays,
        COUNT(*) FILTER (
            WHERE countries_tags IS NOT NULL
              AND len(countries_tags) = 0
        ) AS empty_country_tag_arrays,

        COUNT(*) FILTER (
            WHERE list_contains(countries_tags, 'en:indonesia')
        ) AS rows_tagged_indonesia,
        COUNT(*) FILTER (
            WHERE list_contains(countries_tags, 'en:world')
        ) AS rows_tagged_world,
        COUNT(*) FILTER (
            WHERE len(countries_tags) > 1
        ) AS rows_with_multiple_country_tags,

        COUNT(*) FILTER (
            WHERE completeness IS NULL
        ) AS null_completeness,
        MIN(completeness) AS min_completeness,
        MAX(completeness) AS max_completeness,
        COUNT(*) FILTER (
            WHERE completeness < 0 OR completeness > 1
        ) AS completeness_outside_0_1

    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(
    profile.T.rename(columns={0: "value"})
)


In [ ]:

duplicate_audit = con.execute("""
    WITH code_counts AS (
        SELECT
            code,
            COUNT(*) AS row_count
        FROM read_parquet(?)
        WHERE code IS NOT NULL
          AND TRIM(code) <> ''
        GROUP BY code
    )
    SELECT
        COUNT(*) AS distinct_valid_codes,
        COUNT(*) FILTER (
            WHERE row_count > 1
        ) AS repeated_codes,
        SUM(
            CASE WHEN row_count > 1
                 THEN row_count ELSE 0 END
        ) AS rows_in_repeated_groups,
        SUM(
            CASE WHEN row_count > 1
                 THEN row_count - 1 ELSE 0 END
        ) AS excess_rows_if_one_per_code,
        MAX(row_count) AS max_rows_per_code
    FROM code_counts
""", [local_path]).fetchdf()

display(duplicate_audit.T.rename(columns={0: "value"}))


In [ ]:

duplicate_examples = con.execute("""
    WITH repeated_codes AS (
        SELECT code
        FROM read_parquet(?)
        WHERE code IS NOT NULL
          AND TRIM(code) <> ''
        GROUP BY code
        HAVING COUNT(*) > 1
    )
    SELECT
        p.code,
        p.brands,
        p.brands_tags,
        p.categories_tags,
        p.countries_tags,
        p.created_t,
        p.completeness
    FROM read_parquet(?) AS p
    INNER JOIN repeated_codes AS d
        ON p.code = d.code
    ORDER BY p.code
    LIMIT 20
""", [local_path, local_path]).fetchdf()

display(duplicate_examples)



In [ ]:

exact_duplicate_audit = con.execute("""
    WITH repeated_codes AS (
        SELECT code
        FROM read_parquet(?)
        WHERE code IS NOT NULL
          AND TRIM(code) <> ''
        GROUP BY code
        HAVING COUNT(*) > 1
    ),
    duplicate_rows AS (
        SELECT p.*
        FROM read_parquet(?) AS p
        INNER JOIN repeated_codes AS d
            ON p.code = d.code
    ),
    distinct_rows AS (
        SELECT DISTINCT *
        FROM duplicate_rows
    )
    SELECT
        (SELECT COUNT(*) FROM duplicate_rows)
            AS candidate_rows,
        (SELECT COUNT(*) FROM distinct_rows)
            AS distinct_full_records,
        (SELECT COUNT(*) FROM duplicate_rows)
        - (SELECT COUNT(*) FROM distinct_rows)
            AS exact_duplicate_excess_rows
""", [local_path, local_path]).fetchdf()

display(exact_duplicate_audit)


In [ ]:

country_profile = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND country_tag <> 'en:world'
      AND country_tag <> 'en:null'
    GROUP BY country_tag
    ORDER BY product_records DESC
""", [local_path]).fetchdf()

display(country_profile)


In [ ]:

country_checks = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND (
          lower(country_tag) LIKE '%indonesia%'
          OR lower(country_tag) LIKE '%irland%'
          OR lower(country_tag) LIKE '%middle east%'
          OR lower(country_tag) LIKE '%piemonte%'
          OR lower(country_tag) LIKE '%sark%'
          OR lower(country_tag) LIKE '%alderney%'
          OR lower(country_tag) LIKE '%world%'
          OR lower(country_tag) LIKE '%null%'
      )
    GROUP BY country_tag
    ORDER BY product_records DESC, country_tag
""", [local_path]).fetchdf()

display(country_checks)


In [ ]:

country_candidates = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND lower(country_tag) NOT IN (
          'en:world',
          'en:worldwide',
          'en:middle east',
          'en:piemonte',
          'en:sark',
          'en:alderney'
      )
    GROUP BY country_tag
    ORDER BY product_records DESC
""", [local_path]).fetchdf()

print("Jumlah label setelah pengecualian awal:",
      len(country_candidates))
display(country_candidates)


In [ ]:

language_prefix_audit = con.execute("""
    WITH country_tags AS (
        SELECT
            country_tag,
            regexp_extract(
                country_tag, '^([^:]+):', 1
            ) AS language_prefix,
            COUNT(*) AS product_records
        FROM read_parquet(?)
        CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
        WHERE country_tag IS NOT NULL
        GROUP BY country_tag
    )
    SELECT
        CASE
            WHEN language_prefix = ''
                THEN '(no prefix)'
            ELSE lower(language_prefix)
        END AS language_prefix,
        COUNT(*) AS distinct_labels,
        SUM(product_records) AS tag_assignments
    FROM country_tags
    GROUP BY 1
    ORDER BY distinct_labels DESC
""", [local_path]).fetchdf()

display(language_prefix_audit)


In [ ]:
import os

print("File di /kaggle/working:")
print(os.listdir("/kaggle/working"))

print("\nDirektori input Kaggle:")
print(os.listdir("/kaggle/input"))

In [ ]:
import duckdb

con = duckdb.connect()

local_path = (
    "/kaggle/input/datasets/maulanakinditamami/"
    "open-food-facts-product-database-salin-judul/food.parquet"
)

result = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE code IS NOT NULL) AS non_null_codes
    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(result)

In [ ]:

country_shortlist = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND regexp_matches(country_tag, '^en:')
      AND lower(country_tag) NOT IN (
          'en:world',
          'en:worldwide',
          'en:middle east',
          'en:piemonte',
          'en:sark',
          'en:alderney',
          'en:null'
      )
    GROUP BY country_tag
    HAVING COUNT(*) >= 1000
    ORDER BY product_records DESC
""", [local_path]).fetchdf()

print("Negara/label dengan >= 1.000 penetapan tag:",
      len(country_shortlist))
display(country_shortlist)


In [ ]:
import duckdb
import os

local_path = (
    "/kaggle/input/datasets/maulanakinditamami/"
    "open-food-facts-product-database-salin-judul/food.parquet"
)

assert os.path.exists(local_path), "File dataset tidak ditemukan!"

con = duckdb.connect()

print("DuckDB siap.")
print("Dataset ditemukan:", os.path.exists(local_path))

In [ ]:
country_shortlist = con.execute("""
    SELECT
        country_tag,
        COUNT(*) AS product_records
    FROM read_parquet(?)
    CROSS JOIN UNNEST(countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND regexp_matches(country_tag, '^en:')
      AND lower(country_tag) NOT IN (
          'en:world',
          'en:worldwide',
          'en:middle east',
          'en:piemonte',
          'en:sark',
          'en:alderney',
          'en:null'
      )
    GROUP BY country_tag
    HAVING COUNT(*) >= 1000
    ORDER BY product_records DESC
""", [local_path]).fetchdf()

print("Negara/label dengan >= 1.000 penetapan tag:",
      len(country_shortlist))

display(
    country_shortlist[
        country_shortlist["country_tag"].str.contains(
            "indonesia|malaysia|singapore|thailand|vietnam|philippines",
            case=False,
            na=False
        )
    ]
)

In [ ]:
duplicate_audit = con.execute("""
    WITH code_counts AS (
        SELECT
            code,
            COUNT(*) AS row_count
        FROM read_parquet(?)
        WHERE code IS NOT NULL
          AND TRIM(code) <> ''
        GROUP BY code
    )
    SELECT
        COUNT(*) AS distinct_valid_codes,
        COUNT(*) FILTER (
            WHERE row_count > 1
        ) AS repeated_codes,
        SUM(
            CASE
                WHEN row_count > 1 THEN row_count
                ELSE 0
            END
        ) AS rows_in_repeated_groups,
        SUM(
            CASE
                WHEN row_count > 1 THEN row_count - 1
                ELSE 0
            END
        ) AS excess_rows_if_one_per_code
    FROM code_counts
""", [local_path]).fetchdf()

display(duplicate_audit)

In [ ]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
""", [local_path]).fetchdf()

In [ ]:
relevant_columns = con.execute("""
    DESCRIBE
    SELECT
        code,
        brands,
        brands_tags,
        categories,
        categories_tags,
        countries_tags,
        completeness,
        created_t
    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(relevant_columns)

In [ ]:
array_profile = con.execute("""
    SELECT
        COUNT(*) AS total_records,

        COUNT(*) FILTER (
            WHERE brands_tags IS NULL
        ) AS null_brand_arrays,

        COUNT(*) FILTER (
            WHERE brands_tags IS NOT NULL
              AND len(brands_tags) = 0
        ) AS empty_brand_arrays,

        COUNT(*) FILTER (
            WHERE brands_tags IS NOT NULL
              AND len(brands_tags) > 1
        ) AS multiple_brand_arrays,

        COUNT(*) FILTER (
            WHERE categories_tags IS NOT NULL
              AND len(categories_tags) > 1
        ) AS multiple_category_arrays,

        COUNT(*) FILTER (
            WHERE countries_tags IS NOT NULL
              AND len(countries_tags) > 1
        ) AS multiple_country_arrays
    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(array_profile)

In [ ]:
identity_check = con.execute("""
    SELECT
        COUNT(*) AS total_records,

        COUNT(DISTINCT code) AS distinct_codes,

        COUNT(DISTINCT code) FILTER (
            WHERE code IS NOT NULL
              AND TRIM(code) <> ''
        ) AS distinct_valid_codes,

        COUNT(*) FILTER (
            WHERE code IS NULL
               OR TRIM(code) = ''
        ) AS missing_or_empty_codes,

        COUNT(DISTINCT created_t) AS distinct_created_t,

        COUNT(DISTINCT creator) AS distinct_creators
    FROM read_parquet(?)
""", [local_path]).fetchdf()

display(identity_check)

In [ ]:
# Membuat tabel inti dari snapshot Parquet
con.execute("""
    CREATE OR REPLACE TABLE product_records AS
    SELECT
        ROW_NUMBER() OVER () AS record_id,
        code,
        brands,
        brands_tags,
        categories,
        categories_tags,
        countries_tags,
        completeness,
        created_t
    FROM read_parquet(?)
""", [local_path])

print(
    con.execute("""
        SELECT
            COUNT(*) AS total_records,
            COUNT(DISTINCT record_id) AS unique_record_ids,
            MIN(record_id) AS min_record_id,
            MAX(record_id) AS max_record_id
        FROM product_records
    """).fetchdf()
)

In [ ]:
display(
    con.execute("""
        DESCRIBE product_records
    """).fetchdf()
)

display(
    con.execute("""
        SELECT *
        FROM product_records
        ORDER BY record_id
        LIMIT 10
    """).fetchdf()
)

In [ ]:
import os

sample_path = "/kaggle/working/product_records_sample.csv"

con.execute("""
    COPY (
        SELECT *
        FROM product_records
        ORDER BY record_id
        LIMIT 100
    )
    TO '/kaggle/working/product_records_sample.csv'
    (HEADER, DELIMITER ',')
""")

print("File exists:", os.path.exists(sample_path))
print("File size:", round(os.path.getsize(sample_path) / 1024, 2), "KB")
print("File path:", sample_path)

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE product_countries AS
    SELECT DISTINCT
        p.record_id,
        country_tag
    FROM product_records AS p
    CROSS JOIN UNNEST(p.countries_tags) AS t(country_tag)
    WHERE country_tag IS NOT NULL
      AND TRIM(country_tag) <> ''
""")

display(
    con.execute("""
        SELECT *
        FROM product_countries
        ORDER BY record_id, country_tag
        LIMIT 10
    """).fetchdf()
)

In [ ]:
display(
    con.execute("""
        SELECT
            COUNT(*) AS total_record_country_pairs,
            COUNT(DISTINCT record_id) AS records_with_country_tags,
            COUNT(DISTINCT country_tag) AS distinct_country_tags,
            COUNT(*) - COUNT(DISTINCT record_id) AS additional_country_assignments
        FROM product_countries
    """).fetchdf()
)

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE product_categories AS
    SELECT DISTINCT
        p.record_id,
        category_tag
    FROM product_records AS p
    CROSS JOIN UNNEST(p.categories_tags) AS t(category_tag)
    WHERE category_tag IS NOT NULL
      AND TRIM(category_tag) <> ''
""")

display(
    con.execute("""
        SELECT *
        FROM product_categories
        ORDER BY record_id, category_tag
        LIMIT 10
    """).fetchdf()
)

In [ ]:
display(
    con.execute("""
        SELECT
            COUNT(*) AS total_record_category_pairs,
            COUNT(DISTINCT record_id) AS records_with_category_tags,
            COUNT(DISTINCT category_tag) AS distinct_category_tags,
            COUNT(*) - COUNT(DISTINCT record_id)
                AS additional_category_assignments
        FROM product_categories
    """).fetchdf()
)

In [ ]:
display(
    con.execute("""
        SELECT
            category_tag,
            COUNT(*) AS record_count
        FROM product_categories
        WHERE LOWER(TRIM(category_tag)) IN (
            'en:null',
            'en:unknown',
            'en:undefined'
        )
        GROUP BY category_tag
        ORDER BY record_count DESC
    """).fetchdf()
)

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE product_brands AS
    SELECT DISTINCT
        p.record_id,
        brand_tag
    FROM product_records AS p
    CROSS JOIN UNNEST(p.brands_tags) AS t(brand_tag)
    WHERE brand_tag IS NOT NULL
      AND TRIM(brand_tag) <> ''
""")

display(
    con.execute("""
        SELECT *
        FROM product_brands
        ORDER BY record_id, brand_tag
        LIMIT 10
    """).fetchdf()
)

In [ ]:
display(
    con.execute("""
        SELECT
            COUNT(*) AS total_record_brand_pairs,
            COUNT(DISTINCT record_id) AS records_with_brand_tags,
            COUNT(DISTINCT brand_tag) AS distinct_brand_tags,
            COUNT(*) - COUNT(DISTINCT record_id)
                AS additional_brand_assignments
        FROM product_brands
    """).fetchdf()
)

In [ ]:
display(
    con.execute("""
        SELECT
            p.brands,
            b.brand_tag,
            COUNT(*) AS record_count
        FROM product_brands AS b
        JOIN product_records AS p
            ON b.record_id = p.record_id
        WHERE p.brands IS NOT NULL
          AND TRIM(p.brands) <> ''
        GROUP BY
            p.brands,
            b.brand_tag
        ORDER BY record_count DESC
        LIMIT 30
    """).fetchdf()
)

In [ ]:
display(
    con.execute("""
        WITH normalized AS (
            SELECT
                brand_tag,
                LOWER(
                    REGEXP_REPLACE(
                        TRIM(brand_tag),
                        '^xx:',
                        ''
                    )
                ) AS normalized_tag
            FROM product_brands
        ),
        tag_groups AS (
            SELECT
                normalized_tag,
                COUNT(DISTINCT brand_tag) AS original_tag_variants
            FROM normalized
            GROUP BY normalized_tag
        )
        SELECT
            COUNT(*) AS normalized_tag_groups,
            SUM(original_tag_variants) AS original_distinct_tags,
            COUNT(*) FILTER (
                WHERE original_tag_variants > 1
            ) AS groups_with_multiple_variants,
            SUM(original_tag_variants - 1) AS excess_tag_variants
        FROM tag_groups
    """).fetchdf()
)

In [ ]:

import os

sample_tables = {
    "product_records": "product_records_sample.csv",
    "product_countries": "product_countries_sample.csv",
    "product_categories": "product_categories_sample.csv",
    "product_brands": "product_brands_sample.csv",
}

for table_name, filename in sample_tables.items():
    output_path = f"/kaggle/working/{filename}"

    con.execute(f"""
        COPY (
            SELECT *
            FROM {table_name}
            ORDER BY record_id
            LIMIT 100
        )
        TO '{output_path}'
        (HEADER, DELIMITER ',')
    """)

    print(
        f"{filename}: "
        f"{os.path.getsize(output_path):,} bytes"
    )


In [ ]:

for table_name, filename in sample_tables.items():
    output_path = f"/kaggle/working/{filename}"

    result = con.execute("""
        SELECT COUNT(*) AS sample_rows
        FROM read_csv_auto(?)
    """, [output_path]).fetchdf()

    print(f"\n{table_name}")
    display(result)


In [ ]:
import duckdb
import os

local_path = "/kaggle/input/datasets/maulanakinditamami/open-food-facts-product-database-salin-judul/food.parquet"

assert os.path.exists(local_path), "File Parquet tidak ditemukan"

con = duckdb.connect()
print("DuckDB siap")
print("Dataset ditemukan:", os.path.exists(local_path))

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE product_records AS
SELECT
    ROW_NUMBER() OVER () AS record_id,
    code,
    brands,
    brands_tags,
    categories,
    categories_tags,
    countries_tags,
    completeness,
    created_t
FROM read_parquet(?)
""", [local_path])

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT record_id) AS unique_record_ids,
    MIN(record_id) AS min_record_id,
    MAX(record_id) AS max_record_id
FROM product_records
""").fetchdf()

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE product_countries AS
SELECT DISTINCT
    p.record_id,
    t.country_tag
FROM product_records AS p
CROSS JOIN UNNEST(p.countries_tags) AS t(country_tag)
WHERE t.country_tag IS NOT NULL
  AND TRIM(t.country_tag) <> ''
""")

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE product_categories AS
SELECT DISTINCT
    p.record_id,
    t.category_tag
FROM product_records AS p
CROSS JOIN UNNEST(p.categories_tags) AS t(category_tag)
WHERE t.category_tag IS NOT NULL
  AND TRIM(t.category_tag) <> ''
""")

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE product_brands AS
SELECT DISTINCT
    p.record_id,
    t.brand_tag
FROM product_records AS p
CROSS JOIN UNNEST(p.brands_tags) AS t(brand_tag)
WHERE t.brand_tag IS NOT NULL
  AND TRIM(t.brand_tag) <> ''
""")

In [ ]:
con.execute("""
SELECT 'product_records' AS table_name, COUNT(*) AS row_count
FROM product_records
UNION ALL
SELECT 'product_countries', COUNT(*)
FROM product_countries
UNION ALL
SELECT 'product_categories', COUNT(*)
FROM product_categories
UNION ALL
SELECT 'product_brands', COUNT(*)
FROM product_brands
""").fetchdf()

In [ ]:
con.execute("""
WITH bridge_checks AS (
    SELECT
        'product_countries' AS table_name,
        COUNT(*) AS total_pairs,
        COUNT(DISTINCT record_id) AS records_linked,
        COUNT(*) - COUNT(DISTINCT (record_id, country_tag)) AS duplicate_pairs
    FROM product_countries

    UNION ALL

    SELECT
        'product_categories',
        COUNT(*),
        COUNT(DISTINCT record_id),
        COUNT(*) - COUNT(DISTINCT (record_id, category_tag))
    FROM product_categories

    UNION ALL

    SELECT
        'product_brands',
        COUNT(*),
        COUNT(DISTINCT record_id),
        COUNT(*) - COUNT(DISTINCT (record_id, brand_tag))
    FROM product_brands
)
SELECT *
FROM bridge_checks;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    'product_countries' AS table_name,
    COUNT(*) AS orphan_rows
FROM product_countries AS b
LEFT JOIN product_records AS p
    ON b.record_id = p.record_id
WHERE p.record_id IS NULL

UNION ALL

SELECT
    'product_categories',
    COUNT(*)
FROM product_categories AS b
LEFT JOIN product_records AS p
    ON b.record_id = p.record_id
WHERE p.record_id IS NULL

UNION ALL

SELECT
    'product_brands',
    COUNT(*)
FROM product_brands AS b
LEFT JOIN product_records AS p
    ON b.record_id = p.record_id
WHERE p.record_id IS NULL;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    LOWER(TRIM(category_tag)) AS normalized_tag,
    COUNT(*) AS tag_assignments,
    COUNT(DISTINCT record_id) AS affected_records
FROM product_categories
WHERE LOWER(TRIM(category_tag)) IN (
    'en:null',
    'en:unknown',
    'en:undefined'
)
GROUP BY LOWER(TRIM(category_tag))
ORDER BY tag_assignments DESC;
""").fetchdf()

In [ ]:
con.execute("""
WITH category_quality AS (
    SELECT
        record_id,
        COUNT(*) AS total_category_tags,
        COUNT(*) FILTER (
            WHERE LOWER(TRIM(category_tag)) NOT IN (
                'en:null',
                'en:unknown',
                'en:undefined'
            )
        ) AS valid_category_tags
    FROM product_categories
    GROUP BY record_id
)
SELECT
    COUNT(*) AS records_with_any_category_tag,
    COUNT(*) FILTER (
        WHERE valid_category_tags > 0
    ) AS records_with_valid_categories,
    COUNT(*) FILTER (
        WHERE valid_category_tags = 0
    ) AS records_with_only_invalid_categories
FROM category_quality;
""").fetchdf()

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE product_categories_clean AS
SELECT
    record_id,
    category_tag
FROM product_categories
WHERE LOWER(TRIM(category_tag)) NOT IN (
    'en:null',
    'en:unknown',
    'en:undefined'
);
""")

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_valid_pairs,
    COUNT(DISTINCT record_id) AS records_with_valid_categories,
    COUNT(DISTINCT category_tag) AS distinct_valid_category_tags
FROM product_categories_clean;
""").fetchdf()

In [ ]:
con.execute("""
SELECT COUNT(*) AS remaining_invalid_pairs
FROM product_categories_clean
WHERE category_tag IN (
    'en:null',
    'en:unknown',
    'en:undefined'
);
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    LOWER(TRIM(brand_tag)) AS normalized_tag,
    COUNT(*) AS tag_assignments,
    COUNT(DISTINCT record_id) AS affected_records
FROM product_brands
WHERE brand_tag IS NOT NULL
GROUP BY LOWER(TRIM(brand_tag))
HAVING LOWER(TRIM(brand_tag)) IN (
    'en:null',
    'en:unknown',
    'en:undefined',
    'en:'
)
ORDER BY tag_assignments DESC;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    brand_tag,
    LOWER(TRIM(brand_tag)) AS normalized_tag,
    COUNT(*) AS tag_assignments,
    COUNT(DISTINCT record_id) AS affected_records
FROM product_brands
WHERE brand_tag IS NOT NULL
GROUP BY brand_tag, LOWER(TRIM(brand_tag))
HAVING brand_tag <> LOWER(TRIM(brand_tag))
ORDER BY tag_assignments DESC
LIMIT 30;
""").fetchdf()

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE product_brands_clean AS
SELECT DISTINCT
    record_id,
    LOWER(TRIM(brand_tag)) AS brand_tag
FROM product_brands
WHERE brand_tag IS NOT NULL
  AND TRIM(brand_tag) <> '';
""")

In [ ]:
con.execute("""
SELECT
    (SELECT COUNT(*) FROM product_brands) AS original_pairs,
    (SELECT COUNT(*) FROM product_brands_clean) AS cleaned_pairs,
    (SELECT COUNT(DISTINCT record_id) FROM product_brands_clean)
        AS records_with_brands,
    (SELECT COUNT(DISTINCT brand_tag) FROM product_brands_clean)
        AS distinct_normalized_brands;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_pairs,
    COUNT(DISTINCT record_id || '|' || brand_tag) AS unique_pairs
FROM product_brands_clean;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_pairs,
    COUNT(*) - (
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT record_id, brand_tag
            FROM product_brands_clean
        ) AS unique_pairs
    ) AS duplicate_pairs
FROM product_brands_clean;
""").fetchdf()

In [ ]:
import duckdb
import os

local_path = (
    "/kaggle/input/datasets/maulanakinditamami/"
    "open-food-facts-product-database-salin-judul/food.parquet"
)

assert os.path.exists(local_path), "File dataset tidak ditemukan!"

con = duckdb.connect()

# 1. Core product records
con.execute("""
CREATE OR REPLACE TABLE product_records AS
SELECT
    ROW_NUMBER() OVER () AS record_id,
    code,
    brands,
    brands_tags,
    categories,
    categories_tags,
    countries_tags,
    completeness,
    created_t
FROM read_parquet(?);
""", [local_path])

# 2. Country bridge
con.execute("""
CREATE OR REPLACE TABLE product_countries AS
SELECT DISTINCT
    p.record_id,
    t.country_tag
FROM product_records AS p
CROSS JOIN UNNEST(p.countries_tags) AS t(country_tag)
WHERE t.country_tag IS NOT NULL
  AND TRIM(t.country_tag) <> '';
""")

# 3. Category bridge
con.execute("""
CREATE OR REPLACE TABLE product_categories AS
SELECT DISTINCT
    p.record_id,
    t.category_tag
FROM product_records AS p
CROSS JOIN UNNEST(p.categories_tags) AS t(category_tag)
WHERE t.category_tag IS NOT NULL
  AND TRIM(t.category_tag) <> '';
""")

# 4. Brand bridge
con.execute("""
CREATE OR REPLACE TABLE product_brands AS
SELECT DISTINCT
    p.record_id,
    t.brand_tag
FROM product_records AS p
CROSS JOIN UNNEST(p.brands_tags) AS t(brand_tag)
WHERE t.brand_tag IS NOT NULL
  AND TRIM(t.brand_tag) <> '';
""")

# 5. Clean category bridge
con.execute("""
CREATE OR REPLACE TABLE product_categories_clean AS
SELECT
    record_id,
    category_tag
FROM product_categories
WHERE LOWER(TRIM(category_tag)) NOT IN (
    'en:null',
    'en:unknown',
    'en:undefined'
);
""")

# 6. Clean brand bridge
con.execute("""
CREATE OR REPLACE TABLE product_brands_clean AS
SELECT DISTINCT
    record_id,
    LOWER(TRIM(brand_tag)) AS brand_tag
FROM product_brands
WHERE brand_tag IS NOT NULL
  AND TRIM(brand_tag) <> '';
""")

print("Semua tabel berhasil dibangun.")

In [ ]:
con.execute("""
SELECT 'product_records' AS table_name, COUNT(*) AS row_count
FROM product_records

UNION ALL

SELECT 'product_countries', COUNT(*)
FROM product_countries

UNION ALL

SELECT 'product_categories', COUNT(*)
FROM product_categories

UNION ALL

SELECT 'product_brands', COUNT(*)
FROM product_brands

UNION ALL

SELECT 'product_categories_clean', COUNT(*)
FROM product_categories_clean

UNION ALL

SELECT 'product_brands_clean', COUNT(*)
FROM product_brands_clean;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    country_tag,
    COUNT(*) AS tag_assignments,
    COUNT(DISTINCT record_id) AS product_records
FROM product_countries
GROUP BY country_tag
ORDER BY tag_assignments DESC
LIMIT 50;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    country_tag,
    COUNT(*) AS tag_assignments,
    COUNT(DISTINCT record_id) AS product_records
FROM product_countries
WHERE
    LOWER(TRIM(country_tag)) IN (
        'en:world',
        'en:worldwide',
        'en:international',
        'en:unknown',
        'en:null',
        'en:undefined'
    )
    OR LOWER(TRIM(country_tag)) NOT LIKE 'en:%'
GROUP BY country_tag
ORDER BY tag_assignments DESC;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    country_tag,
    COUNT(DISTINCT record_id) AS product_records
FROM product_countries
WHERE LOWER(TRIM(country_tag)) IN (
    'en:indonesia',
    'en:malaysia',
    'en:singapore',
    'en:thailand',
    'en:philippines',
    'en:vietnam',
    'en:united-states',
    'en:france',
    'en:germany',
    'en:united-kingdom',
    'en:japan',
    'en:china',
    'en:india',
    'en:world',
    'en:worldwide'
)
GROUP BY country_tag
ORDER BY product_records DESC;
""").fetchdf()

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE country_tag_mapping AS
SELECT *
FROM (
    VALUES
        ('en:france', 'France', 'Europe'),
        ('en:united-states', 'United States', 'North America'),
        ('en:germany', 'Germany', 'Europe'),
        ('en:united-kingdom', 'United Kingdom', 'Europe'),
        ('en:japan', 'Japan', 'East Asia'),
        ('en:india', 'India', 'South Asia'),
        ('en:singapore', 'Singapore', 'Southeast Asia'),
        ('en:thailand', 'Thailand', 'Southeast Asia'),
        ('en:philippines', 'Philippines', 'Southeast Asia'),
        ('en:indonesia', 'Indonesia', 'Southeast Asia'),
        ('en:malaysia', 'Malaysia', 'Southeast Asia'),
        ('en:china', 'China', 'East Asia'),
        ('en:vietnam', 'Vietnam', 'Southeast Asia')
) AS t(country_tag, country_name, region);
""")

In [ ]:
country_results = con.execute("""
SELECT
    m.country_name,
    m.region,
    COUNT(DISTINCT pc.record_id) AS recorded_product_records
FROM product_countries AS pc
JOIN country_tag_mapping AS m
    ON LOWER(TRIM(pc.country_tag)) = m.country_tag
GROUP BY
    m.country_name,
    m.region
ORDER BY recorded_product_records DESC;
""").fetchdf()

country_results

In [ ]:
country_sensitivity = con.execute("""
SELECT
    m.country_name,
    COUNT(DISTINCT pc.record_id) AS record_count,
    COUNT(DISTINCT CASE
        WHEN p.code IS NOT NULL
         AND TRIM(p.code) <> ''
        THEN p.code
    END) AS unique_valid_codes
FROM product_countries AS pc
JOIN country_tag_mapping AS m
    ON LOWER(TRIM(pc.country_tag)) = m.country_tag
JOIN product_records AS p
    ON p.record_id = pc.record_id
GROUP BY m.country_name
ORDER BY record_count DESC;
""").fetchdf()

country_sensitivity 

In [ ]:
con.execute("""
SELECT
    m.country_name,
    m.region,
    COUNT(DISTINCT pc.record_id) AS recorded_product_records
FROM product_countries AS pc
JOIN country_tag_mapping AS m
    ON LOWER(TRIM(pc.country_tag)) = m.country_tag
GROUP BY
    m.country_name,
    m.region
ORDER BY recorded_product_records DESC;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    m.country_name,
    COUNT(DISTINCT pc.record_id) AS record_count,
    COUNT(DISTINCT CASE
        WHEN p.code IS NOT NULL
         AND TRIM(p.code) <> ''
        THEN p.code
    END) AS unique_valid_codes
FROM product_countries AS pc
JOIN country_tag_mapping AS m
    ON LOWER(TRIM(pc.country_tag)) = m.country_tag
JOIN product_records AS p
    ON p.record_id = pc.record_id
GROUP BY m.country_name
ORDER BY record_count DESC;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    country_tag,
    country_name
FROM country_tag_mapping
WHERE LOWER(TRIM(country_tag)) IN (
    'en:world',
    'en:worldwide'
);
""").fetchdf()

In [ ]:
con.execute("""
SELECT COUNT(*) AS mapped_country_count
FROM country_tag_mapping;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    category_tag,
    COUNT(DISTINCT record_id) AS recorded_product_records
FROM product_categories_clean
GROUP BY category_tag
ORDER BY recorded_product_records DESC
LIMIT 20;
""").fetchdf()

In [ ]:
con.execute("""
WITH top_categories AS (
    SELECT
        category_tag,
        COUNT(DISTINCT record_id) AS recorded_product_records
    FROM product_categories_clean
    GROUP BY category_tag
    ORDER BY recorded_product_records DESC
    LIMIT 20
)
SELECT
    tc.category_tag,
    tc.recorded_product_records,
    COUNT(DISTINCT pb.record_id) AS records_with_brand,
    COUNT(DISTINCT pb.brand_tag) AS distinct_brand_labels,
    ROUND(
        100.0 * COUNT(DISTINCT pb.record_id)
        / NULLIF(tc.recorded_product_records, 0),
        2
    ) AS records_with_brand_pct
FROM top_categories AS tc
LEFT JOIN product_categories_clean AS pc
    ON pc.category_tag = tc.category_tag
LEFT JOIN product_brands_clean AS pb
    ON pb.record_id = pc.record_id
GROUP BY
    tc.category_tag,
    tc.recorded_product_records
ORDER BY tc.recorded_product_records DESC;
""").fetchdf()

In [ ]:
con.execute("""
WITH top_categories AS (
    SELECT
        category_tag,
        COUNT(DISTINCT record_id) AS recorded_product_records
    FROM product_categories_clean
    GROUP BY category_tag
    ORDER BY recorded_product_records DESC
    LIMIT 20
),
category_brand_metrics AS (
    SELECT
        tc.category_tag,
        tc.recorded_product_records,
        COUNT(DISTINCT pb.record_id) AS records_with_brand,
        COUNT(DISTINCT pb.brand_tag) AS distinct_brand_labels
    FROM top_categories AS tc
    JOIN product_categories_clean AS pc
        ON pc.category_tag = tc.category_tag
    LEFT JOIN product_brands_clean AS pb
        ON pb.record_id = pc.record_id
    GROUP BY
        tc.category_tag,
        tc.recorded_product_records
)
SELECT
    *,
    ROUND(
        100.0 * records_with_brand
        / NULLIF(recorded_product_records, 0),
        2
    ) AS records_with_brand_pct,
    ROUND(
        1.0 * distinct_brand_labels
        / NULLIF(records_with_brand, 0),
        4
    ) AS distinct_brand_labels_per_branded_record
FROM category_brand_metrics
ORDER BY recorded_product_records DESC;
""").fetchdf()

In [ ]:
result = con.execute("""
WITH top_categories AS (
    SELECT
        category_tag,
        COUNT(DISTINCT record_id) AS recorded_product_records
    FROM product_categories_clean
    GROUP BY category_tag
    ORDER BY recorded_product_records DESC
    LIMIT 20
),
category_metrics AS (
    SELECT
        tc.category_tag,
        tc.recorded_product_records,
        COUNT(DISTINCT pb.record_id) AS records_with_brand
    FROM top_categories AS tc
    JOIN product_categories_clean AS pc
        ON pc.category_tag = tc.category_tag
    LEFT JOIN product_brands_clean AS pb
        ON pb.record_id = pc.record_id
    GROUP BY
        tc.category_tag,
        tc.recorded_product_records
)
SELECT
    *,
    ROUND(
        100.0 * records_with_brand
        / NULLIF(recorded_product_records, 0),
        2
    ) AS records_with_brand_pct
FROM category_metrics
WHERE records_with_brand > recorded_product_records
   OR records_with_brand < 0
   OR 100.0 * records_with_brand
      / NULLIF(recorded_product_records, 0) > 100;
""").fetchdf()

result

In [ ]:
con.execute("""
WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),
brand_counts AS (
    SELECT
        ir.record_id,
        COUNT(DISTINCT pb.brand_tag) AS brand_label_count
    FROM indonesia_records AS ir
    LEFT JOIN product_brands_clean AS pb
        ON pb.record_id = ir.record_id
    GROUP BY ir.record_id
)
SELECT
    COUNT(*) AS indonesia_record_count,
    COUNT(*) FILTER (
        WHERE brand_label_count > 0
    ) AS records_with_brand,
    COUNT(*) FILTER (
        WHERE brand_label_count = 0
    ) AS records_without_brand,
    COUNT(*) FILTER (
        WHERE brand_label_count = 1
    ) AS records_with_one_brand,
    COUNT(*) FILTER (
        WHERE brand_label_count > 1
    ) AS records_with_multiple_brands,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE brand_label_count > 0
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS brand_coverage_pct
FROM brand_counts;
""").fetchdf()

In [ ]:
result = con.execute("""
WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),
single_brand_records AS (
    SELECT
        ir.record_id,
        MIN(pb.brand_tag) AS brand_tag
    FROM indonesia_records AS ir
    JOIN product_brands_clean AS pb
        ON pb.record_id = ir.record_id
    GROUP BY ir.record_id
    HAVING COUNT(DISTINCT pb.brand_tag) = 1
),
brand_distribution AS (
    SELECT
        brand_tag,
        COUNT(*) AS recorded_product_records
    FROM single_brand_records
    GROUP BY brand_tag
),
ranked_brands AS (
    SELECT
        brand_tag,
        recorded_product_records,
        ROUND(
            100.0 * recorded_product_records
            / SUM(recorded_product_records) OVER (),
            2
        ) AS record_pct,
        ROW_NUMBER() OVER (
            ORDER BY recorded_product_records DESC, brand_tag
        ) AS brand_rank
    FROM brand_distribution
)
SELECT
    brand_rank,
    brand_tag,
    recorded_product_records,
    record_pct,
    ROUND(
        SUM(record_pct) OVER (
            ORDER BY brand_rank
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS cumulative_record_pct
FROM ranked_brands
ORDER BY brand_rank
LIMIT 20;
""").fetchdf()

result

In [ ]:
result = con.execute("""
WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),
single_brand_records AS (
    SELECT
        ir.record_id,
        MIN(pb.brand_tag) AS brand_tag
    FROM indonesia_records AS ir
    JOIN product_brands_clean AS pb
        ON pb.record_id = ir.record_id
    GROUP BY ir.record_id
    HAVING COUNT(DISTINCT pb.brand_tag) = 1
),
brand_distribution AS (
    SELECT
        brand_tag,
        COUNT(*) AS recorded_product_records
    FROM single_brand_records
    GROUP BY brand_tag
),
ranked_brands AS (
    SELECT
        brand_tag,
        recorded_product_records,
        ROW_NUMBER() OVER (
            ORDER BY recorded_product_records DESC, brand_tag
        ) AS brand_rank
    FROM brand_distribution
)
SELECT
    COUNT(*) AS distinct_brand_labels,
    SUM(recorded_product_records) AS records_in_analysis,
    MAX(CASE WHEN brand_rank = 1
        THEN recorded_product_records END) AS top_brand_records,
    ROUND(
        100.0 * SUM(CASE WHEN brand_rank <= 5
            THEN recorded_product_records ELSE 0 END)
        / SUM(recorded_product_records),
        2
    ) AS top_5_record_concentration_pct,
    ROUND(
        100.0 * SUM(CASE WHEN brand_rank <= 10
            THEN recorded_product_records ELSE 0 END)
        / SUM(recorded_product_records),
        2
    ) AS top_10_record_concentration_pct,
    ROUND(
        100.0 * SUM(CASE WHEN brand_rank <= 20
            THEN recorded_product_records ELSE 0 END)
        / SUM(recorded_product_records),
        2
    ) AS top_20_record_concentration_pct
FROM ranked_brands;
""").fetchdf()

result

In [ ]:

sql_script = """
-- 05_brand_analysis.sql
-- Purpose:
-- Analyze brand-label coverage, distribution, and record concentration
-- among Open Food Facts records tagged as Indonesia.
--
-- Source tables:
--   product_countries
--   product_brands_clean
--
-- Definitions:
--   Indonesia records are records tagged en:indonesia.
--   Brand coverage is the percentage of Indonesia records with at least
--   one cleaned brand label.
--   Brand distribution and concentration use records with exactly one
--   cleaned brand label to avoid allocating one record to multiple labels.
--
-- Limitations:
--   Record counts are not sales, market size, or market share.
--   Country tags do not prove country of manufacture or sales.
--   Brand labels may contain aliases or represent different entities.
--   The analysis excludes records with multiple brand labels from
--   the distribution and concentration calculations.


-- 1. Brand coverage profile for Indonesia

WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),
brand_counts AS (
    SELECT
        ir.record_id,
        COUNT(DISTINCT pb.brand_tag) AS brand_label_count
    FROM indonesia_records AS ir
    LEFT JOIN product_brands_clean AS pb
        ON pb.record_id = ir.record_id
    GROUP BY ir.record_id
)
SELECT
    COUNT(*) AS indonesia_record_count,
    COUNT(*) FILTER (
        WHERE brand_label_count > 0
    ) AS records_with_brand,
    COUNT(*) FILTER (
        WHERE brand_label_count = 0
    ) AS records_without_brand,
    COUNT(*) FILTER (
        WHERE brand_label_count = 1
    ) AS records_with_one_brand,
    COUNT(*) FILTER (
        WHERE brand_label_count > 1
    ) AS records_with_multiple_brands,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE brand_label_count > 0
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS brand_coverage_pct
FROM brand_counts;


-- 2. Top 20 brand labels by recorded product records
-- Denominator: Indonesia records with exactly one cleaned brand label.

WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),
single_brand_records AS (
    SELECT
        ir.record_id,
        MIN(pb.brand_tag) AS brand_tag
    FROM indonesia_records AS ir
    JOIN product_brands_clean AS pb
        ON pb.record_id = ir.record_id
    GROUP BY ir.record_id
    HAVING COUNT(DISTINCT pb.brand_tag) = 1
),
brand_distribution AS (
    SELECT
        brand_tag,
        COUNT(*) AS recorded_product_records
    FROM single_brand_records
    GROUP BY brand_tag
),
ranked_brands AS (
    SELECT
        brand_tag,
        recorded_product_records,
        ROW_NUMBER() OVER (
            ORDER BY recorded_product_records DESC, brand_tag
        ) AS brand_rank
    FROM brand_distribution
)
SELECT
    brand_rank,
    brand_tag,
    recorded_product_records,
    ROUND(
        100.0 * recorded_product_records
        / SUM(recorded_product_records) OVER (),
        2
    ) AS record_pct,
    ROUND(
        100.0 * SUM(recorded_product_records) OVER (
            ORDER BY brand_rank
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) / SUM(recorded_product_records) OVER (),
        2
    ) AS cumulative_record_pct
FROM ranked_brands
WHERE brand_rank <= 20
ORDER BY brand_rank;


-- 3. Top 5, Top 10, and Top 20 record concentration
-- Concentration is calculated from unrounded record counts.
-- Denominator: all Indonesia records with exactly one cleaned brand label.

WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),
single_brand_records AS (
    SELECT
        ir.record_id,
        MIN(pb.brand_tag) AS brand_tag
    FROM indonesia_records AS ir
    JOIN product_brands_clean AS pb
        ON pb.record_id = ir.record_id
    GROUP BY ir.record_id
    HAVING COUNT(DISTINCT pb.brand_tag) = 1
),
brand_distribution AS (
    SELECT
        brand_tag,
        COUNT(*) AS recorded_product_records
    FROM single_brand_records
    GROUP BY brand_tag
),
ranked_brands AS (
    SELECT
        brand_tag,
        recorded_product_records,
        ROW_NUMBER() OVER (
            ORDER BY recorded_product_records DESC, brand_tag
        ) AS brand_rank
    FROM brand_distribution
)
SELECT
    COUNT(*) AS distinct_brand_labels,
    SUM(recorded_product_records) AS records_in_analysis,
    MAX(
        CASE WHEN brand_rank = 1
             THEN recorded_product_records
        END
    ) AS top_brand_records,
    ROUND(
        100.0 * SUM(
            CASE WHEN brand_rank <= 5
                 THEN recorded_product_records ELSE 0
            END
        ) / SUM(recorded_product_records),
        2
    ) AS top_5_record_concentration_pct,
    ROUND(
        100.0 * SUM(
            CASE WHEN brand_rank <= 10
                 THEN recorded_product_records ELSE 0
            END
        ) / SUM(recorded_product_records),
        2
    ) AS top_10_record_concentration_pct,
    ROUND(
        100.0 * SUM(
            CASE WHEN brand_rank <= 20
                 THEN recorded_product_records ELSE 0
            END
        ) / SUM(recorded_product_records),
        2
    ) AS top_20_record_concentration_pct
FROM ranked_brands;
"""

# Jalankan ketiga query berurutan dan tampilkan masing-masing hasil.
statements = [
    statement.strip()
    for statement in sql_script.split(";")
    if statement.strip()
    and not all(
        line.strip().startswith("--") or not line.strip()
        for line in statement.splitlines()
    )
]

for i, statement in enumerate(statements, start=1):
    print(f"QUERY {i}")
    display(con.execute(statement).fetchdf())


In [ ]:
result = con.execute("""
WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),
single_brand_records AS (
    SELECT
        ir.record_id,
        MIN(pb.brand_tag) AS brand_tag
    FROM indonesia_records AS ir
    JOIN product_brands_clean AS pb
        ON pb.record_id = ir.record_id
    GROUP BY ir.record_id
    HAVING COUNT(DISTINCT pb.brand_tag) = 1
),
brand_distribution AS (
    SELECT
        brand_tag,
        COUNT(*) AS recorded_product_records
    FROM single_brand_records
    GROUP BY brand_tag
),
ranked_brands AS (
    SELECT
        brand_tag,
        recorded_product_records,
        ROW_NUMBER() OVER (
            ORDER BY recorded_product_records DESC, brand_tag
        ) AS brand_rank,
        SUM(recorded_product_records) OVER () AS total_records
    FROM brand_distribution
)
SELECT
    brand_rank,
    brand_tag,
    recorded_product_records,
    ROUND(
        100.0 * recorded_product_records
        / NULLIF(total_records, 0),
        2
    ) AS record_pct,
    ROUND(
        100.0 * SUM(recorded_product_records) OVER (
            ORDER BY brand_rank
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) / NULLIF(total_records, 0),
        2
    ) AS cumulative_record_pct
FROM ranked_brands
WHERE brand_rank <= 20
ORDER BY brand_rank;
""").fetchdf()

result

In [ ]:

result = con.execute("""
SELECT
    'product_records' AS table_name,
    COUNT(*) AS row_count
FROM product_records

UNION ALL

SELECT
    'product_countries',
    COUNT(*)
FROM product_countries

UNION ALL

SELECT
    'product_categories_clean',
    COUNT(*)
FROM product_categories_clean

UNION ALL

SELECT
    'product_brands_clean',
    COUNT(*)
FROM product_brands_clean;
""").df()

display(result)


In [ ]:

validation_result = con.execute("""
SELECT
    'record_id_null' AS check_name,
    COUNT(*) AS issue_count
FROM product_records
WHERE record_id IS NULL

UNION ALL

SELECT
    'record_id_duplicate',
    COUNT(*) - COUNT(DISTINCT record_id)
FROM product_records

UNION ALL

SELECT
    'country_bridge_null_or_blank_tag',
    COUNT(*)
FROM product_countries
WHERE country_tag IS NULL
   OR TRIM(country_tag) = ''

UNION ALL

SELECT
    'country_bridge_duplicate_pairs',
    COUNT(*) - COUNT(DISTINCT (record_id, country_tag))
FROM product_countries

UNION ALL

SELECT
    'clean_category_invalid_tags',
    COUNT(*)
FROM product_categories_clean
WHERE category_tag IS NULL
   OR TRIM(category_tag) = ''
   OR LOWER(TRIM(category_tag)) IN (
       'en:null',
       'en:unknown',
       'en:undefined'
   )

UNION ALL

SELECT
    'clean_category_duplicate_pairs',
    COUNT(*) - COUNT(DISTINCT (record_id, category_tag))
FROM product_categories_clean

UNION ALL

SELECT
    'clean_brand_null_or_blank_tags',
    COUNT(*)
FROM product_brands_clean
WHERE brand_tag IS NULL
   OR TRIM(brand_tag) = ''

UNION ALL

SELECT
    'clean_brand_duplicate_pairs',
    COUNT(*) - COUNT(DISTINCT (record_id, brand_tag))
FROM product_brands_clean

ORDER BY check_name;
""").df()

display(validation_result)


In [ ]:

metric_validation = con.execute("""
WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),

brand_counts AS (
    SELECT
        ir.record_id,
        COUNT(DISTINCT pb.brand_tag) AS brand_label_count
    FROM indonesia_records AS ir
    LEFT JOIN product_brands_clean AS pb
        ON ir.record_id = pb.record_id
    GROUP BY ir.record_id
),

brand_coverage AS (
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE brand_label_count > 0
        ) AS records_with_brand,
        COUNT(*) FILTER (
            WHERE brand_label_count = 0
        ) AS records_without_brand,
        COUNT(*) FILTER (
            WHERE brand_label_count = 1
        ) AS single_brand_records
    FROM brand_counts
),

single_brand_distribution AS (
    SELECT
        pb.brand_tag,
        COUNT(DISTINCT ir.record_id) AS record_count
    FROM indonesia_records AS ir
    JOIN product_brands_clean AS pb
        ON ir.record_id = pb.record_id
    GROUP BY ir.record_id, pb.brand_tag
    HAVING COUNT(DISTINCT pb.brand_tag) = 1
),

ranked_brands AS (
    SELECT
        brand_tag,
        record_count,
        ROW_NUMBER() OVER (
            ORDER BY record_count DESC, brand_tag
        ) AS brand_rank
    FROM single_brand_distribution
),

concentration AS (
    SELECT
        COUNT(*) AS distinct_brand_labels,
        SUM(record_count) AS total_records,
        SUM(CASE WHEN brand_rank <= 5
                 THEN record_count ELSE 0 END) AS top_5_records,
        SUM(CASE WHEN brand_rank <= 10
                 THEN record_count ELSE 0 END) AS top_10_records,
        SUM(CASE WHEN brand_rank <= 20
                 THEN record_count ELSE 0 END) AS top_20_records
    FROM ranked_brands
),

checks AS (
    SELECT
        'indonesia_brand_coverage_reconciliation' AS check_name,
        CASE
            WHEN bc.total_records =
                 bc.records_with_brand + bc.records_without_brand
            THEN 0 ELSE 1
        END AS issue_count
    FROM brand_coverage AS bc

    UNION ALL

    SELECT
        'single_brand_count_not_above_branded_count',
        CASE
            WHEN bc.single_brand_records <= bc.records_with_brand
            THEN 0 ELSE 1
        END
    FROM brand_coverage AS bc

    UNION ALL

    SELECT
        'single_brand_distribution_reconciliation',
        CASE
            WHEN c.total_records = bc.single_brand_records
            THEN 0 ELSE 1
        END
    FROM brand_coverage AS bc
    CROSS JOIN concentration AS c

    UNION ALL

    SELECT
        'top_5_not_above_top_10',
        CASE
            WHEN c.top_5_records <= c.top_10_records
            THEN 0 ELSE 1
        END
    FROM concentration AS c

    UNION ALL

    SELECT
        'top_10_not_above_top_20',
        CASE
            WHEN c.top_10_records <= c.top_20_records
            THEN 0 ELSE 1
        END
    FROM concentration AS c

    UNION ALL

    SELECT
        'top_20_not_above_total',
        CASE
            WHEN c.top_20_records <= c.total_records
            THEN 0 ELSE 1
        END
    FROM concentration AS c
)

SELECT *
FROM checks
ORDER BY check_name;

""").df()

display(metric_validation)

In [ ]:

distribution_check = con.execute("""
WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),

brand_counts AS (
    SELECT
        ir.record_id,
        COUNT(DISTINCT pb.brand_tag) AS brand_label_count
    FROM indonesia_records AS ir
    LEFT JOIN product_brands_clean AS pb
        ON ir.record_id = pb.record_id
    GROUP BY ir.record_id
),

brand_coverage AS (
    SELECT
        COUNT(*) FILTER (
            WHERE brand_label_count = 1
        ) AS single_brand_records
    FROM brand_counts
),

single_brand_records AS (
    SELECT
        ir.record_id,
        MIN(pb.brand_tag) AS brand_tag
    FROM indonesia_records AS ir
    INNER JOIN product_brands_clean AS pb
        ON ir.record_id = pb.record_id
    GROUP BY ir.record_id
    HAVING COUNT(DISTINCT pb.brand_tag) = 1
),

single_brand_distribution AS (
    SELECT
        brand_tag,
        COUNT(*) AS record_count
    FROM single_brand_records
    GROUP BY brand_tag
),

concentration AS (
    SELECT
        SUM(record_count) AS total_records
    FROM single_brand_distribution
)

SELECT
    bc.single_brand_records,
    c.total_records,
    bc.single_brand_records - c.total_records AS difference,
    CASE
        WHEN bc.single_brand_records = c.total_records THEN 0
        ELSE 1
    END AS issue_count
FROM brand_coverage AS bc
CROSS JOIN concentration AS c;
""").df()

display(distribution_check)


In [ ]:

metric_validation = con.execute("""
WITH indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),

brand_counts AS (
    SELECT
        ir.record_id,
        COUNT(DISTINCT pb.brand_tag) AS brand_label_count
    FROM indonesia_records AS ir
    LEFT JOIN product_brands_clean AS pb
        ON ir.record_id = pb.record_id
    GROUP BY ir.record_id
),

brand_coverage AS (
    SELECT
        COUNT(*) AS total_indonesia_records,
        COUNT(*) FILTER (
            WHERE brand_label_count > 0
        ) AS records_with_brand,
        COUNT(*) FILTER (
            WHERE brand_label_count = 0
        ) AS records_without_brand,
        COUNT(*) FILTER (
            WHERE brand_label_count = 1
        ) AS single_brand_records
    FROM brand_counts
),

single_brand_records AS (
    SELECT
        ir.record_id,
        MIN(pb.brand_tag) AS brand_tag
    FROM indonesia_records AS ir
    INNER JOIN product_brands_clean AS pb
        ON ir.record_id = pb.record_id
    GROUP BY ir.record_id
    HAVING COUNT(DISTINCT pb.brand_tag) = 1
),

single_brand_distribution AS (
    SELECT
        brand_tag,
        COUNT(*) AS record_count
    FROM single_brand_records
    GROUP BY brand_tag
),

ranked_brands AS (
    SELECT
        brand_tag,
        record_count,
        ROW_NUMBER() OVER (
            ORDER BY record_count DESC, brand_tag
        ) AS brand_rank
    FROM single_brand_distribution
),

concentration AS (
    SELECT
        COUNT(*) AS distinct_brand_labels,
        SUM(record_count) AS total_records,
        SUM(CASE WHEN brand_rank <= 5
                 THEN record_count ELSE 0 END) AS top_5_records,
        SUM(CASE WHEN brand_rank <= 10
                 THEN record_count ELSE 0 END) AS top_10_records,
        SUM(CASE WHEN brand_rank <= 20
                 THEN record_count ELSE 0 END) AS top_20_records
    FROM ranked_brands
),

checks AS (
    SELECT
        'indonesia_brand_coverage_reconciliation' AS check_name,
        CASE
            WHEN total_indonesia_records =
                 records_with_brand + records_without_brand
            THEN 0 ELSE 1
        END AS issue_count
    FROM brand_coverage

    UNION ALL

    SELECT
        'single_brand_count_not_above_branded_count',
        CASE
            WHEN single_brand_records <= records_with_brand
            THEN 0 ELSE 1
        END
    FROM brand_coverage

    UNION ALL

    SELECT
        'single_brand_distribution_reconciliation',
        CASE
            WHEN bc.single_brand_records = c.total_records
            THEN 0 ELSE 1
        END
    FROM brand_coverage AS bc
    CROSS JOIN concentration AS c

    UNION ALL

    SELECT
        'top_5_not_above_top_10',
        CASE WHEN top_5_records <= top_10_records
             THEN 0 ELSE 1 END
    FROM concentration

    UNION ALL

    SELECT
        'top_10_not_above_top_20',
        CASE WHEN top_10_records <= top_20_records
             THEN 0 ELSE 1 END
    FROM concentration

    UNION ALL

    SELECT
        'top_20_not_above_total',
        CASE WHEN top_20_records <= total_records
             THEN 0 ELSE 1 END
    FROM concentration
)

SELECT *
FROM checks
ORDER BY check_name;
""").df()

display(metric_validation)


In [ ]:

validation_result = con.execute("""
WITH
indonesia_records AS (
    SELECT DISTINCT record_id
    FROM product_countries
    WHERE LOWER(TRIM(country_tag)) = 'en:indonesia'
),

brand_counts AS (
    SELECT
        ir.record_id,
        COUNT(DISTINCT pb.brand_tag) AS brand_label_count
    FROM indonesia_records AS ir
    LEFT JOIN product_brands_clean AS pb
        ON ir.record_id = pb.record_id
    GROUP BY ir.record_id
),

brand_coverage AS (
    SELECT
        COUNT(*) AS total_indonesia_records,
        COUNT(*) FILTER (
            WHERE brand_label_count > 0
        ) AS records_with_brand,
        COUNT(*) FILTER (
            WHERE brand_label_count = 0
        ) AS records_without_brand,
        COUNT(*) FILTER (
            WHERE brand_label_count = 1
        ) AS single_brand_records
    FROM brand_counts
),

single_brand_records AS (
    SELECT
        ir.record_id,
        MIN(pb.brand_tag) AS brand_tag
    FROM indonesia_records AS ir
    INNER JOIN product_brands_clean AS pb
        ON ir.record_id = pb.record_id
    GROUP BY ir.record_id
    HAVING COUNT(DISTINCT pb.brand_tag) = 1
),

single_brand_distribution AS (
    SELECT
        brand_tag,
        COUNT(*) AS record_count
    FROM single_brand_records
    GROUP BY brand_tag
),

ranked_brands AS (
    SELECT
        brand_tag,
        record_count,
        ROW_NUMBER() OVER (
            ORDER BY record_count DESC, brand_tag
        ) AS brand_rank
    FROM single_brand_distribution
),

concentration AS (
    SELECT
        COUNT(*) AS distinct_brand_labels,
        SUM(record_count) AS total_records,
        SUM(
            CASE WHEN brand_rank <= 5
                 THEN record_count ELSE 0 END
        ) AS top_5_records,
        SUM(
            CASE WHEN brand_rank <= 10
                 THEN record_count ELSE 0 END
        ) AS top_10_records,
        SUM(
            CASE WHEN brand_rank <= 20
                 THEN record_count ELSE 0 END
        ) AS top_20_records
    FROM ranked_brands
),

integrity_checks AS (
    SELECT
        'record_id_null' AS check_name,
        COUNT(*) AS issue_count
    FROM product_records
    WHERE record_id IS NULL

    UNION ALL

    SELECT
        'record_id_duplicate',
        COUNT(*) - COUNT(DISTINCT record_id)
    FROM product_records

    UNION ALL

    SELECT
        'country_bridge_null_or_blank_tag',
        COUNT(*)
    FROM product_countries
    WHERE country_tag IS NULL
       OR TRIM(country_tag) = ''

    UNION ALL

    SELECT
        'country_bridge_duplicate_pairs',
        COUNT(*) - COUNT(DISTINCT (record_id, country_tag))
    FROM product_countries

    UNION ALL

    SELECT
        'clean_category_invalid_tags',
        COUNT(*)
    FROM product_categories_clean
    WHERE category_tag IS NULL
       OR TRIM(category_tag) = ''
       OR LOWER(TRIM(category_tag)) IN (
           'en:null',
           'en:unknown',
           'en:undefined'
       )

    UNION ALL

    SELECT
        'clean_category_duplicate_pairs',
        COUNT(*) - COUNT(DISTINCT (record_id, category_tag))
    FROM product_categories_clean

    UNION ALL

    SELECT
        'clean_brand_null_or_blank_tags',
        COUNT(*)
    FROM product_brands_clean
    WHERE brand_tag IS NULL
       OR TRIM(brand_tag) = ''

    UNION ALL

    SELECT
        'clean_brand_duplicate_pairs',
        COUNT(*) - COUNT(DISTINCT (record_id, brand_tag))
    FROM product_brands_clean
),

metric_checks AS (
    SELECT
        'indonesia_brand_coverage_reconciliation' AS check_name,
        CASE
            WHEN total_indonesia_records =
                 records_with_brand + records_without_brand
            THEN 0 ELSE 1
        END AS issue_count
    FROM brand_coverage

    UNION ALL

    SELECT
        'single_brand_count_not_above_branded_count',
        CASE
            WHEN single_brand_records <= records_with_brand
            THEN 0 ELSE 1
        END
    FROM brand_coverage

    UNION ALL

    SELECT
        'single_brand_distribution_reconciliation',
        CASE
            WHEN bc.single_brand_records = c.total_records
            THEN 0 ELSE 1
        END
    FROM brand_coverage AS bc
    CROSS JOIN concentration AS c

    UNION ALL

    SELECT
        'top_5_not_above_top_10',
        CASE
            WHEN top_5_records <= top_10_records
            THEN 0 ELSE 1
        END
    FROM concentration

    UNION ALL

    SELECT
        'top_10_not_above_top_20',
        CASE
            WHEN top_10_records <= top_20_records
            THEN 0 ELSE 1
        END
    FROM concentration

    UNION ALL

    SELECT
        'top_20_not_above_total',
        CASE
            WHEN top_20_records <= total_records
            THEN 0 ELSE 1
        END
    FROM concentration
)

SELECT check_name, issue_count
FROM integrity_checks

UNION ALL

SELECT check_name, issue_count
FROM metric_checks

ORDER BY check_name;
""").df()

display(validation_result)

assert len(validation_result) == 14, (
    f"Jumlah pemeriksaan tidak sesuai: {len(validation_result)}"
)

assert (validation_result["issue_count"] == 0).all(), (
    "Ada pemeriksaan validasi yang gagal. "
    "Periksa tabel validation_result."
)

print("PASS: Seluruh 14 pemeriksaan validasi berhasil.")


In [1]:
import os

local_path = (
    "/kaggle/input/datasets/maulanakinditamami/"
    "open-food-facts-product-database-salin-judul/food.parquet"
)

print("=== DATASET ===")
print("File ditemukan:", os.path.isfile(local_path))

if os.path.isfile(local_path):
    size_bytes = os.path.getsize(local_path)
    print(f"Ukuran file: {size_bytes / (1024**3):.2f} GiB")

print("\n=== FILE DI /kaggle/working ===")
working_dir = "/kaggle/working"

for root, dirs, files in os.walk(working_dir):
    depth = root.replace(working_dir, "").count(os.sep)

    if depth >= 2:
        dirs[:] = []
        continue

    for filename in files:
        path = os.path.join(root, filename)
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"{path} | {size_mb:.2f} MiB")

=== DATASET ===
File ditemukan: True
Ukuran file: 7.43 GiB

=== FILE DI /kaggle/working ===


In [2]:
print("Variabel con tersedia:", "con" in globals())

if "con" in globals():
    try:
        print(con.execute("SHOW TABLES").fetchall())
    except Exception as e:
        print(type(e).__name__, str(e))

Variabel con tersedia: False


In [3]:
import os

search_dirs = [
    "/kaggle/working",
    "/kaggle/temp",
    "/tmp",
]

extensions = (".duckdb", ".db", ".sqlite", ".csv", ".parquet")

print("=== FILE HASIL YANG MUNGKIN BISA DIPULIHKAN ===")

for base in search_dirs:
    if not os.path.isdir(base):
        continue

    for root, dirs, files in os.walk(base):
        # Batasi pencarian agar tidak terlalu lama.
        depth = root.replace(base, "").count(os.sep)
        if depth >= 3:
            dirs[:] = []
            continue

        for filename in files:
            if filename.lower().endswith(extensions):
                path = os.path.join(root, filename)
                size_mb = os.path.getsize(path) / (1024 ** 2)
                print(f"{path} | {size_mb:.2f} MiB")
                

=== FILE HASIL YANG MUNGKIN BISA DIPULIHKAN ===
